# Chapter 4 — Vision-Language Model (VLM) from Scratch

**Goal**: Connect the ViT (Ch 1) and GPT (Ch 2) with a learned projection,
creating a model that can answer questions about images.

## Architecture (LLaVA-1.5 style)

```
                     ┌─── Frozen ViT ────────────────────┐
 Image (224×224×3) → │ PatchEmbed → 12×ViTBlock → N×D_v  │
                     └───────────────────────────────────┘
                                          ↓
                     ┌─── Trained Projection MLP ─────────┐
                     │ Linear(D_v, D_l) → GELU → Linear   │
                     └───────────────────────────────────┘
                                          ↓  visual tokens (N_img, D_l)
 Text prompt tokens ──────────────────────┤
                                          ↓  concat: [visual | text]
                     ┌─── GPT Decoder ────────────────────┐
                     │ 12×[CausalAttn + FFN] → LM Head    │
                     └───────────────────────────────────┘
                                          ↓
                                   Generated text
```

## Two training stages

| Stage | What trains | Data | Epochs |
|-------|-------------|------|--------|
| 1: Feature alignment | Projection MLP only | Image-caption pairs | 1 |
| 2: Instruction tuning | Projection + LLM | Visual Q&A | 3 |

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn as nn

## 4.1 Projection MLP

The projection layer is small but critical.
It translates vision features (in ViT's semantic space) into language features
(in GPT's semantic space).

In [ ]:
from multimodal_from_scratch.multimodal.projection import ProjectionMLP, LinearProjection

# Create a projection from ViT-Base (768) to GPT-Small (768)
proj_mlp    = ProjectionMLP(vision_dim=768, language_dim=768)
proj_linear = LinearProjection(vision_dim=768, language_dim=768)

# Compare parameter counts
mlp_params    = sum(p.numel() for p in proj_mlp.parameters())
linear_params = sum(p.numel() for p in proj_linear.parameters())

print(f"2-layer MLP projection: {mlp_params:,} parameters")
print(f"Linear projection:      {linear_params:,} parameters")

# Test
visual_tokens = torch.randn(2, 196, 768)  # (B, n_patches, D_v)
projected = proj_mlp(visual_tokens)
print(f"\nInput:  {visual_tokens.shape}")
print(f"Output: {projected.shape}   (same shape, different space)")

## 4.2 Full VLM

In [ ]:
from multimodal_from_scratch.vision.vit import ViTConfig
from multimodal_from_scratch.language.gpt import GPTConfig
from multimodal_from_scratch.multimodal.vlm import VisionLanguageModel, VLMConfig

# Tiny model for CPU testing
cfg = VLMConfig(
    vit=ViTConfig(img_size=64, patch_size=16, embed_dim=192, depth=3, num_heads=3),
    gpt=GPTConfig(vocab_size=1000, context_len=128, embed_dim=192, depth=3,
                  num_heads=3, dropout=0.0),
    image_token_mode='all',   # use all patch tokens
)

vlm = VisionLanguageModel(cfg)

# Count parameters per component
params = vlm.count_parameters(trainable_only=False)
for k, v in params.items():
    print(f"  {k:<20}: {v:>10,}")

## 4.3 Training Stage 1 — Feature Alignment

In [ ]:
# Set stage 1: freeze ViT + LLM, train only projection
vlm.set_stage1()

params_stage1 = vlm.count_parameters(trainable_only=True)
print("Stage 1 — trainable parameters:")
for k, v in params_stage1.items():
    status = '✓ TRAINS' if v > 0 else '✗ frozen'
    print(f"  {k:<20}: {v:>10,}  {status}")

## 4.4 Training Stage 2 — Instruction Tuning

In [ ]:
# Set stage 2: freeze ViT, train projection + LLM
vlm.set_stage2()

params_stage2 = vlm.count_parameters(trainable_only=True)
print("Stage 2 — trainable parameters:")
for k, v in params_stage2.items():
    status = '✓ TRAINS' if v > 0 else '✗ frozen'
    print(f"  {k:<20}: {v:>10,}  {status}")

## 4.5 Forward Pass with Loss

In [ ]:
# Simulate a training batch
B = 2
T = 20

images    = torch.randn(B, 3, 64, 64)
input_ids = torch.randint(0, 1000, (B, T))

# Labels: -100 for image tokens (ignored in loss), actual tokens for text
# Simulate: first 5 tokens are question (masked), last 15 are answer (supervised)
labels = input_ids.clone()
labels[:, :5] = -100    # question tokens — don't compute loss here

out = vlm(images, input_ids, labels)

print(f"Loss: {out['loss'].item():.3f}")
print(f"Logits shape: {out['logits'].shape}")

n_img = cfg.vit.n_patches
print(f"  = (batch={B}, n_img_tokens={n_img} + text_tokens={T}, vocab={1000})")

## 4.6 Image-Conditioned Text Generation

In [ ]:
# Generate text given an image + text prompt
# (untrained model will generate gibberish, but the pipeline is correct)

image  = torch.randn(1, 3, 64, 64)   # single image
prompt = torch.tensor([[1, 2, 3, 4]])  # question token ids

generated = vlm.generate(
    images=image,
    input_ids=prompt,
    max_new_tokens=10,
    temperature=0.8,
    top_k=10,
)

print(f"Prompt:    {prompt.tolist()[0]}")
print(f"Generated: {generated.tolist()[0]}")
print(f"\nWith a trained model + real tokenizer, this would be:")
print(f"  Prompt:    'What is in the image?'")
print(f"  Generated: 'A cat sitting on a mat near a window.'")

## Summary

We have assembled all components into a working VLM:

| Component | Parameters (tiny) | Role |
|-----------|------------------|------|
| ViT-Tiny encoder | ~5M | Image → patch tokens |
| Projection MLP | ~150K | Vision space → language space |
| GPT-Small decoder | ~12M | Language generation |

The projection MLP is only ~1% of total parameters, but it does the
critical work of bridging two pre-trained models.

**Next**: Chapter 5 — Pre-training and Fine-tuning